# Planars — {{LANG_ID}} Annotation Status

Shows the current span results for **{{LANG_ID}}** based on filled annotation sheets.

**To use:** From the menu: **Runtime → Run all** — Google will ask for permission once.

In [ ]:
# Setup — installs planars, authenticates with Google, loads manifest
import subprocess, sys, importlib, importlib.metadata

subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '--pre',
     'planars', 'gspread', 'google-auth'])
importlib.invalidate_caches()
_ver = importlib.metadata.version('planars')
print(f"planars {_ver} \u2713")

from google.colab import auth
auth.authenticate_user()

import json, gspread
from google.auth import default
from googleapiclient.discovery import build
from planars.charts import collect_all_spans_from_sheets, domain_chart

creds, _ = default(scopes=[
    'https://www.googleapis.com/auth/drive',
    'https://www.googleapis.com/auth/spreadsheets',
])
gc = gspread.authorize(creds)
drive_svc = build('drive', 'v3', credentials=creds)

# CONFIG_FILE_ID and LANG_ID are baked in at notebook generation time.
# To update them, re-run: python -m coding generate-notebooks --apply
CONFIG_FILE_ID = '{{CONFIG_FILE_ID}}'
LANG_ID = '{{LANG_ID}}'

config = json.loads(drive_svc.files().get_media(fileId=CONFIG_FILE_ID).execute())
folder_id = config[LANG_ID]['folder_id']

files = drive_svc.files().list(
    q=f"'{folder_id}' in parents and name='manifest_{LANG_ID}.json' and trashed=false",
    fields='files(id)'
).execute()['files']
if not files:
    raise FileNotFoundError(f'No manifest found for {LANG_ID} in Drive folder')
manifest = {LANG_ID: json.loads(drive_svc.files().get_media(fileId=files[0]['id']).execute())}

print(f'Connected. Computing spans for {LANG_ID}...')
df, lang_meta = collect_all_spans_from_sheets(gc, manifest)
meta = lang_meta[LANG_ID]
print(f'Done. {len(df)} spans computed.')

In [ ]:
# Domain chart — hover over segments for details
domain_chart(df, meta['keystone_pos'], meta['pos_to_name'])